# DSA 4 · 01 Generate my block (everyone, Day 1 evening)

Settings: **GPU T4**, **Internet on**. Add Input → **dsa4-data**.

1. Set `BLOCK` below (Naveena 1, Manimegalai 2, Meet 3, Vijaya 4).
2. Run the setup cell and the **smoke test** (20 questions) interactively. Post the printed seconds per question in the team chat.
3. Comment out the smoke test, then **Save Version → Save & Run All** for the full block.
4. Next morning: Output → **New Dataset** `dsa4-block-<BLOCK>` (later runs: New Version).

In [ ]:
# Setup: clone the team repository and install what Kaggle lacks
import os, subprocess, sys
REPO_URL = "https://github.com/YOUR-USERNAME/dsa4-reuse-retrieve-generate.git"
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "bitsandbytes>=0.43"], check=True)
os.chdir("/kaggle/working/repo")
sys.path.insert(0, "src")
print("ready")

In [ ]:
BLOCK = 1      # <- change to your block
DATA = "/kaggle/input/dsa4-data"
OUT = f"/kaggle/working/block{BLOCK}"
MODEL = "Qwen/Qwen2.5-7B-Instruct"   # fallback: "Qwen/Qwen2.5-3B-Instruct"
# If you are resuming, add your own dsa4-block-<BLOCK> dataset as input and set:
RESUME = ""  # e.g. f"/kaggle/input/dsa4-block-{BLOCK}/block{BLOCK}.jsonl"

In [ ]:
# Smoke test: 20 questions. Check the answers are short and the variants look right.
!python scripts/generate_block.py --block {BLOCK} --data {DATA} --out /kaggle/working/smoke --model {MODEL} --limit 20
import pandas as pd
sm = pd.read_json(f"/kaggle/working/smoke/block{BLOCK}.jsonl", lines=True)
pd.set_option("display.max_colwidth", 80)
sm[["qid"]].assign(direct=sm["direct"].str["answer"], rag=sm["rag"].str["answer"],
                   lookalike=sm["variants"].str["lookalike"])

In [ ]:
# Full run for the whole block (resumable)
resume = f"--resume-from {RESUME}" if RESUME else ""
!python scripts/generate_block.py --block {BLOCK} --data {DATA} --out {OUT} --model {MODEL} {resume}